# An-Ra V4 — Kaggle TPU v5e-8 training

This notebook is a separate XLA path for Kaggle's **TPU v5e-8**. It does not use CUDA, `DataParallel`, or the GPU notebook. It resumes one verified V4 full-resume checkpoint, trains across all eight TPU cores with BF16 autocast, and atomically replaces one output checkpoint (`anra-v4-tpu-latest.pt`).

Before **Run All**: select **Accelerator → TPU v5e-8**, enable Internet, and attach the dataset/input containing the checkpoint and training text. The notebook never silently falls back to CPU/GPU.

In [ ]:
# 1. TPU runtime preflight — fail closed if Kaggle did not grant a TPU.
import importlib.util, os, subprocess, sys
from pathlib import Path
os.environ['PJRT_DEVICE'] = 'TPU'
if importlib.util.find_spec('torch_xla') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_xla[tpu]'], check=True)
import torch
import torch_xla.core.xla_model as xm
world_size = xm.xrt_world_size()
device = xm.xla_device()
assert world_size == 8, f'Expected TPU v5e-8 (8 cores), got world_size={world_size}'
print({'device': str(device), 'world_size': world_size, 'torch': torch.__version__, 'xla': getattr(__import__('torch_xla'), '__version__', 'unknown')})

In [ ]:
# 2. Clone the exact training branch used by this notebook.
import os, subprocess, sys
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'core-vnext'
REPO = Path('/kaggle/working/anra')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', f'origin/{REPO_REF}'], check=True)
sys.path.insert(0, str(REPO))
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
os.environ['ANRA_SOURCE_COMMIT'] = SOURCE_COMMIT
print({'repo': str(REPO), 'commit': SOURCE_COMMIT, 'branch': REPO_REF})

In [ ]:
# 3. Locate and verify the latest checkpoint and tokenized continuation pack.
import hashlib, json, os, shutil, tarfile, torch
from anra_core.checkpoint import load_core_checkpoint
INPUT_ROOT = Path('/kaggle/input')
CHECKPOINT_OVERRIDE = os.environ.get('ANRA_TPU_CHECKPOINT')
DATASET_OVERRIDE = os.environ.get('ANRA_TPU_DATASET')
def checkpoint_step(path):
    try:
        payload = torch.load(path, map_location='cpu', weights_only=True)
        return int(payload.get('global_step', payload.get('step', -1))) if isinstance(payload, dict) else -1
    except Exception:
        return -1
checkpoint_candidates = [Path(CHECKPOINT_OVERRIDE)] if CHECKPOINT_OVERRIDE else [p for p in INPUT_ROOT.rglob('*.pt') if p.name in {'anra-v4-current-full-resume.pt', 'anra-v4-latest.pt'} or p.name.startswith('anra-v4-step-')]
checkpoint_candidates = [p for p in checkpoint_candidates if p.is_file()]
if not checkpoint_candidates:
    raise FileNotFoundError('Attach a Kaggle input containing an An-Ra full-resume .pt checkpoint.')
CHECKPOINT = max(checkpoint_candidates, key=lambda p: (checkpoint_step(p), p.stat().st_mtime_ns))
EXPECTED_RESUME_STEP = int(os.environ.get('ANRA_EXPECTED_RESUME_STEP', '21400'))
def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as bundle:
        root = destination.resolve()
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        bundle.extractall(destination)
    train_dirs = [p for p in destination.rglob('train') if p.is_dir() and any(p.glob('*.npy'))]
    if not train_dirs:
        raise FileNotFoundError('continuation archive has no train/*.npy token shards')
    return train_dirs[0].parent
DATASET_CANDIDATES = [Path(DATASET_OVERRIDE)] if DATASET_OVERRIDE else [p for p in INPUT_ROOT.rglob('*') if p.is_file() and (p.name in {'anra_training.txt', 'anra_training.jsonl'} or p.name.endswith('.tar.gz'))]
DATASET_CANDIDATES = [p for p in DATASET_CANDIDATES if p.is_file()]
if not DATASET_CANDIDATES:
    raise FileNotFoundError('Attach a Kaggle input containing anra_training.txt or set ANRA_TPU_DATASET.')
DATASET_SOURCE = DATASET_CANDIDATES[0]
DATASET = safe_extract(DATASET_SOURCE, Path('/kaggle/working/ANRA_TPU_DATA')) if DATASET_SOURCE.name.endswith('.tar.gz') else DATASET_SOURCE
model, metadata, identity = load_core_checkpoint(CHECKPOINT)
assert identity.tokenizer_contract_verified, 'Checkpoint tokenizer contract is not verified.'
actual_step = int(identity.global_step or 0)
if actual_step < EXPECTED_RESUME_STEP:
    raise RuntimeError(f'checkpoint is step {actual_step:,}; waiting for a protected checkpoint at or beyond step {EXPECTED_RESUME_STEP:,}')
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
print({'checkpoint': str(CHECKPOINT), 'step': actual_step, 'checkpoint_sha256': sha256(CHECKPOINT), 'dataset_source': str(DATASET_SOURCE), 'dataset': str(DATASET), 'source_commit': identity.source_commit})

In [ ]:
# 4. TPU training controls. Edit only these values before Run All.
# The preflight above refuses to start unless the protected checkpoint is at or beyond step 21,400.
MAX_STEPS = 1_000_000          # duration is the primary stop condition
MAX_MINUTES = 450              # set 0 to use MAX_STEPS only
BATCH_SIZE_PER_CORE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
SAVE_INTERVAL = 200            # exactly one output file is replaced
LOG_INTERVAL = 10
SEED = 1301
OUTPUT_DIR = Path('/kaggle/working/ANRA_TPU_EXPORT')
OUTPUT_CHECKPOINT = OUTPUT_DIR / 'anra-v4-tpu-latest.pt'
print({'resume_min_step': EXPECTED_RESUME_STEP, 'max_steps': MAX_STEPS, 'max_minutes': MAX_MINUTES, 'batch_per_core': BATCH_SIZE_PER_CORE, 'global_effective_sequences': BATCH_SIZE_PER_CORE * GRAD_ACCUM_STEPS * 8, 'output': str(OUTPUT_CHECKPOINT)})

In [ ]:
# 5. Launch the eight-core XLA trainer. Do not run this cell twice.
import subprocess, sys
command = [sys.executable, '-m', 'training.train_xla', '--dataset-path', str(DATASET), '--output-checkpoint', str(OUTPUT_CHECKPOINT), '--resume-from', str(CHECKPOINT), '--max-steps', str(MAX_STEPS), '--max-minutes', str(MAX_MINUTES), '--batch-size', str(BATCH_SIZE_PER_CORE), '--grad-accum-steps', str(GRAD_ACCUM_STEPS), '--learning-rate', str(LEARNING_RATE), '--save-interval', str(SAVE_INTERVAL), '--log-interval', str(LOG_INTERVAL), '--seed', str(SEED)]
subprocess.run(command, cwd=REPO, env=os.environ.copy(), check=True)

In [ ]:
# 6. Verify the single protected output before downloading it from Kaggle.
assert OUTPUT_CHECKPOINT.is_file(), OUTPUT_CHECKPOINT
result = torch.load(OUTPUT_CHECKPOINT, map_location='cpu', weights_only=True)
assert result.get('checkpoint_artifact_class') == 'full_resume'
assert isinstance(result.get('optimizer_state_dict'), dict)
print({'output': str(OUTPUT_CHECKPOINT), 'bytes': OUTPUT_CHECKPOINT.stat().st_size, 'global_step': result.get('global_step'), 'sha256': sha256(OUTPUT_CHECKPOINT), 'status': 'verified'})